# CorePromoter model energy vs. natural promoter conservation (E. coli)

這個 notebook 拿 `Model_CorePromoter_clean.ipynb` 訓練出來的 CorePromoter model，
把它學到的「各位置鹼基 energy」跟真實 E. coli promoter 的位置保守度做關聯分析。

## 為什麼可以這樣比

`CorePromoterModel` 的 `conv2` 權重是**固定**的：它只是把 `conv1` 的 8 個 8-bp filter
擺在固定位移上相加（3 個 channel 對應 spacer 16/17/18）。所以整個模型的 energy 是
嚴格可加的，可以無損拆成一個 4 × L 的 **energy matrix** `e(i, b)`：

```
E(sequence) = Σ_i e(i, base_i) + conv2.bias[channel]
```

## 座標系統：以 −10 為基準

相對座標 0 = −10 hexamer 的第一個鹼基。`conv2` 把 −35 filter 固定在 position 22，
所以以 −10 對齊時 UP 與 −35 會隨 spacer 平移，spacer 下游則不動。
本分析**只用 spacer = 17**（E. coli 最大宗），對應 conv2 channel 1：

| 元件 | 相對座標 |
|---|---|
| UP (filter 0, 1) | −42 … −27 |
| −35 (filter 2) | −23 … −16 |
| spacer (filter 3, 4) | −14 … +1 |
| −10 3' 半段 + discriminator (filter 5) | +2 … +9 |
| ITS / downstream (filter 6, 7) | +10 … +25 |

8 個 filter × 8 bp = **64 個位置**（−42…+25 之間有 4 個未覆蓋的空隙：−26、−25、−24、−15）。

### kernel 編號與 spacer 自由度

程式裡的 `filter` 欄是 **0-based**（`f0`…`f7`）；口語上的「第 N 個 kernel」是 1-based，
兩者差 1。以下用 1-based 敘述。

conv2 的 tap 位置在三個 channel 之間只有一處不同：

| kernel | #1 | #2 | #3 (−35) | #4 | #5 | #6 | #7 | #8 |
|---|---|---|---|---|---|---|---|---|
| tap (sp16) | 3 | 11 | 22 | 30 | 38 | 46 | 54 | 62 |
| tap (sp17) | 3 | 11 | 22 | 31 | 39 | 47 | 55 | 63 |
| tap (sp18) | 3 | 11 | 22 | 32 | 40 | 48 | 56 | 64 |

**#1–#3 固定，#4–#8 整組同步滑動**，所以唯一隨 spacer 改變的空隙在 **#3 → #4 之間**
（0 / 1 / 2 bp）。#2 → #3 之間另有一個固定的 3 bp 空隙，與 spacer 無關。

因為那個可變空隙在 #5 的**上游**，#4 以後的 kernel 相對 −10 的 register 是**固定的**：
−10 的第一個鹼基永遠落在 **kernel #5 的第 7 格**（0-based `k=6`），三個 spacer 都一樣。
初始化 motif 也是這樣排的：`ATGGGG|TA` + `TAAT|TTTT` → `TA` + `TAAT` = **TATAAT**。
第 5 節結尾有一段 assert 會用天然共識序列自動驗證這個錨點。

## 兩個保守度指標

對每個相對位置 i，從對齊後的真實 promoter 算出鹼基頻率 `f_b(i)`：

- **Information content**（均勻背景）：`IC(i) = 2 − H(i)`，`H(i) = −Σ_b f_b log2 f_b`
- **KL divergence**（基因體背景 `p_b`）：`KL(i) = Σ_b f_b log2(f_b / p_b)`

`p_b` 由 `genomes/NC_000913.2.gb` 全基因體鹼基組成算得。

> **注意**：E. coli 的基因體組成非常接近均勻（GC 50.8%），
> 所以 `KL(i) ≈ IC(i)`，兩者差距只有 ~0.01 bits 等級，圖 1 與圖 2 會長得幾乎一樣。
> 這個設計要延伸到 49 物種時才會顯出差別（例如 *Streptomyces* GC ~72%、*Mycoplasma* GC ~25%）。
> 若想讓 KL 在 E. coli 也有鑑別力，把 `BACKGROUND_SOURCE` 改成 `"flank"`
> （改用 promoter 上游側翼序列當背景）。

## 資料來源

`tables/Data_S1_20250826.xlsx` 的 `Es.co`（1865 筆，有基因註解）與
`Es.co$`（4357 筆，完整 TSS 清單）兩張 sheet 都算，互相對照。

表格裡 `UP + Minus35 + Spacer + Minus10 + Dis + Start + ITR` 串起來就是**連續的基因體序列**
（抽驗 300 筆，跨接點的 120 bp 探針有 98% / 96.7% 能在基因體中原樣找到），
所以不需要用座標回查基因體。

> 表格座標**對不上** `genomes/NC_000913.2.gb`（`Es.co$` 標的是 NC_000913.**3**，
> 兩版之間有 indel 差異，−10 座標比對只有 ~50% / ~38% 命中）。
> 因此基因體檔案在這裡**只用來算背景鹼基組成**，序列一律取自表格。

## 輸出

**三張圖**，全部是 2 列（sheet）× 5 欄（元件）的拆解版面，
用來比較模型對各元件學到的能量與天然保守度的關聯強弱差異：

- **圖 1／圖 2（Position vs Position）**：一個點 = 一個位置。
  x = `Σ_b |e(i,b) − mean_b e(i,·)|`（區辨力），y = IC / KL。看的是**位置重要性**。
- **圖 3（Nucleotide vs Nucleotide）**：一個點 = (位置, 鹼基)。x = 中心化 energy，
  y = log2 enrichment（不加權的 `log2(f_b(i)/p_b)`）。看的是**方向性**：模型偏好的鹼基跟
  天然富集/貧化的鹼基一不一致？

**表**：`outputs/` 下的 per-position、per-base 明細表，以及相關係數彙總表
（含 per-element 分項、bootstrap 95% CI，以及 base 層級的 between/within 分解）

In [ ]:
## Setup
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from Bio import SeqIO
from scipy.stats import pearsonr, spearmanr, permutation_test

SCRIPT_DIR = Path.cwd()
if not (SCRIPT_DIR / "recursive_corepromoter_design.py").exists():
    for base in [SCRIPT_DIR, *SCRIPT_DIR.parents]:
        cand = base / "MS2_Data_PyTorch" / "scripts"
        if (cand / "recursive_corepromoter_design.py").exists():
            SCRIPT_DIR = cand
            break
if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

import recursive_corepromoter_design as legacy

MS2_DIR = legacy.MS2_DIR
TABLE_DIR = legacy.TABLE_DIR
WEIGHTS_DIR = legacy.WEIGHTS_DIR
GENOME_DIR = MS2_DIR / "genomes"
FIG_DIR = MS2_DIR / "figures"
OUT_DIR = MS2_DIR / "outputs"
FIG_DIR.mkdir(exist_ok=True)
OUT_DIR.mkdir(exist_ok=True)

DATA_S1 = TABLE_DIR / "Data_S1_20250826.xlsx"
CHECKPOINT = WEIGHTS_DIR / "weights_CorePromoter_clean.pt"
GENOME_GB = GENOME_DIR / "NC_000913.2.gb"  # Escherichia coli K-12 MG1655

# --- analysis settings ---
SPACER = 17                     # 只分析 spacer=17（conv2 channel 1）
SHEETS = ["Es.co", "Es.co$"]    # 兩張 E. coli sheet 都算並比較
BACKGROUND_SOURCE = "genome"    # "genome" = 全基因體組成; "flank" = promoter 上游側翼組成
FLANK_BG_WIDTH = 50             # BACKGROUND_SOURCE="flank" 時取 UP 欄位最前面幾 bp
CORE_RANGE = (-23, 9)           # 相關係數彙總表裡「core promoter only」子集的相對座標範圍

# --- model settings ---
RETRAIN = False                 # True = 依 clean notebook 流程重訓（需要 tables 裡的 PL/SL/DL/UL/ITS pkl）
RETRAIN_EPOCHS = 50

BASES = ["A", "C", "G", "T"]
BASE_COLORS = {"A": "#2ca02c", "C": "#1f77b4", "G": "#ff7f0e", "T": "#d62728"}
ELEMENT_ORDER = ["UP", "-35", "spacer", "-10", "DIS/downstream"]
ELEMENT_COLORS = {
    "UP": "#937860",
    "-35": "#4c72b0",
    "spacer": "#b8b8b8",
    "-10": "#c44e52",
    "DIS/downstream": "#dd8452",
}

M35_LEN = 6
M10_LEN = 6
M35_FILTER_IDX = 2              # conv1 filter index 2 = TTGACATT (−35)
SPACER_CHANNEL = {16: 0, 17: 1, 18: 2}

device = torch.device("cpu")
print(f"MS2_DIR   : {MS2_DIR}")
print(f"spacer    : {SPACER}  (conv2 channel {SPACER_CHANNEL[SPACER]})")
print(f"background: {BACKGROUND_SOURCE}")

## 1. 載入模型

預設直接載入 `weights/weights_CorePromoter_clean.pt`。
把 `RETRAIN` 設成 `True` 就會改用 `recursive_corepromoter_design.train_core_model()`
重跑一次 clean notebook 的訓練流程（同樣的資料組裝、同樣的 50 epoch）。

In [ ]:
def load_core_model():
    if RETRAIN:
        print(f"Retraining CorePromoter model for {RETRAIN_EPOCHS} epochs ...")
        model, history, _ = legacy.train_core_model(epochs=RETRAIN_EPOCHS, device=device)
        print(f"final train/test MSE (log10): "
              f"{history['train_mse_log10'].iloc[-1]:.4f} / {history['test_mse_log10'].iloc[-1]:.4f}")
        return model, {"source": "retrained in-notebook", "epochs": RETRAIN_EPOCHS}

    ckpt = legacy.torch_load_weights(CHECKPOINT, device)
    model = legacy.CorePromoterModel(
        seq_length=int(ckpt["seq_length"]),
        num_conds=int(ckpt["num_conds"]),
    ).to(device)
    model.load_state_dict(ckpt["model_state_dict"], strict=True)
    meta = {k: v for k, v in ckpt.items() if not hasattr(v, "shape") and k != "model_state_dict"}
    print(f"Loaded {CHECKPOINT.name}")
    for k, v in meta.items():
        print(f"  {k}: {v}")
    return model.eval(), meta


model, model_meta = load_core_model()
model.eval()
print(f"\nconv1.weight {tuple(model.conv1.weight.shape)}  (filter, base, position)")
print(f"conv2.weight {tuple(model.conv2.weight.shape)}  (channel, filter, position)")

## 2. 把模型拆成以 −10 為基準的 energy matrix

從 `conv2` 讀出每個 filter 的固定位移，換算成相對 −10 的座標，
再把 `conv1.weight[filter, base, k]` 攤平成 `e(rel_pos, base)`。

In [ ]:
def conv2_filter_offsets(core_model, channel):
    """每個 conv1 filter 在 conv2 kernel 上的固定位移。"""
    w2 = core_model.conv2.weight.detach().cpu().numpy()[channel]  # (n_filters, 65)
    offsets = []
    for f in range(w2.shape[0]):
        nz = np.flatnonzero(np.abs(w2[f]) > 1e-6)
        if nz.size != 1:
            raise ValueError(f"conv2 channel {channel} filter {f} has {nz.size} taps, expected 1")
        offsets.append(int(nz[0]))
    return offsets


def annotate_element(rel, spacer):
    if 0 <= rel < M10_LEN:
        return "-10"
    if -spacer <= rel < 0:
        return "spacer"
    if -(spacer + M35_LEN) <= rel < -spacer:
        return "-35"
    if rel < -(spacer + M35_LEN):
        return "UP"
    return "DIS/downstream"


def build_energy_matrix(core_model, spacer):
    """回傳以 −10 hexamer 第一個鹼基為 rel_pos=0 的 energy matrix。"""
    channel = SPACER_CHANNEL[spacer]
    offsets = conv2_filter_offsets(core_model, channel)
    m10_start = offsets[M35_FILTER_IDX] + M35_LEN + spacer
    w1 = core_model.conv1.weight.detach().cpu().numpy()  # (n_filters, 4, k)

    rows = []
    for f, off in enumerate(offsets):
        for k in range(w1.shape[2]):
            rows.append({
                "rel_pos": off + k - m10_start,
                "filter": f,
                "filter_pos": k,
                **{b: float(w1[f, bi, k]) for bi, b in enumerate(BASES)},
            })

    em = pd.DataFrame(rows).sort_values("rel_pos").reset_index(drop=True)
    if not em["rel_pos"].is_unique:
        raise ValueError("conv2 filter footprints overlap; energy matrix is not additive per position")

    em["element"] = [annotate_element(r, spacer) for r in em["rel_pos"]]
    # 位置重要性指標。先對每個位置置中：同一位置四個鹼基一起加常數 c，等於每條序列的
    # energy 都 +c（被 conv2.bias / param_e0 吸收），是不影響鹼基區辨的 gauge 自由度，
    # 不置中就會把它算進「重要性」裡。
    centered = em[BASES].sub(em[BASES].mean(axis=1), axis=0)
    em["energy_absum"] = centered.abs().sum(axis=1)   # 主指標：Σ_b |e(i,b) − mean_b|
    em["energy_span"] = em[BASES].max(axis=1) - em[BASES].min(axis=1)   # 參考：只看最極端兩個鹼基
    em["energy_sd"] = em[BASES].std(axis=1, ddof=0)                     # 參考：L2 版本
    em["energy_mean"] = em[BASES].mean(axis=1)
    return em, offsets, m10_start


energy_mat, filter_offsets, model_m10_start = build_energy_matrix(model, SPACER)

print(f"conv2 filter offsets (channel {SPACER_CHANNEL[SPACER]}): {filter_offsets}")
print(f"−35 start = {filter_offsets[M35_FILTER_IDX]}, −10 start = {model_m10_start}")
print(f"modelled positions: {len(energy_mat)}  "
      f"(rel {energy_mat['rel_pos'].min()} … {energy_mat['rel_pos'].max()})")
missing = sorted(set(range(energy_mat["rel_pos"].min(), energy_mat["rel_pos"].max() + 1))
                 - set(energy_mat["rel_pos"]))
print(f"uncovered gaps: {missing}")
print()
print(energy_mat.groupby("element", sort=False).agg(
    n_pos=("rel_pos", "size"),
    rel_min=("rel_pos", "min"),
    rel_max=("rel_pos", "max"),
    mean_absum=("energy_absum", "mean"),
).reindex([e for e in ELEMENT_ORDER if e in set(energy_mat["element"])]).round(3))

## 3. 讀入真實 E. coli promoter 並以 −10 對齊

In [ ]:
FIELDS = ["UP", "Minus35", "Spacer", "Minus10", "Dis", "Start", "ITR"]


def load_promoters(sheet, spacer):
    raw = pd.read_excel(DATA_S1, sheet_name=sheet)
    n_raw = len(raw)

    d = raw.dropna(subset=FIELDS).copy()
    n_complete = len(d)
    for f in FIELDS:
        d[f] = d[f].astype(str).str.upper().str.strip()

    d = d[d["Spacer"].str.len() == spacer]
    d = d[(d["Minus35"].str.len() == M35_LEN) & (d["Minus10"].str.len() == M10_LEN)]

    d["aligned"] = d[FIELDS].agg("".join, axis=1)
    d["m10_index"] = d["UP"].str.len() + M35_LEN + spacer

    stats = {
        "sheet": sheet,
        "n_raw": n_raw,
        "n_complete": n_complete,
        "n_spacer": len(d),
    }
    return d.reset_index(drop=True), stats


def position_counts(promoters, rel_positions):
    """對齊後每個相對位置的 A/C/G/T 計數。"""
    rel_min, rel_max = int(min(rel_positions)), int(max(rel_positions))
    width = rel_max - rel_min + 1

    windows = []
    for seq, m10 in zip(promoters["aligned"].to_numpy(), promoters["m10_index"].to_numpy()):
        start, stop = int(m10) + rel_min, int(m10) + rel_max + 1
        if start >= 0 and stop <= len(seq):
            windows.append(seq[start:stop])
    if not windows:
        raise ValueError("no promoter is long enough to cover the requested window")

    arr = np.array(windows, dtype=f"S{width}").view("S1").reshape(len(windows), width)
    counts = np.stack([(arr == b.encode()).sum(axis=0) for b in BASES], axis=1)

    out = pd.DataFrame(counts, columns=BASES)
    out.insert(0, "rel_pos", np.arange(rel_min, rel_max + 1))
    out["n_acgt"] = out[BASES].sum(axis=1)
    return out, len(windows)


rel_positions = energy_mat["rel_pos"].to_numpy()
promoter_sets, count_tables, load_stats = {}, {}, []

for sheet in SHEETS:
    prom, stats = load_promoters(sheet, SPACER)
    counts, n_used = position_counts(prom, rel_positions)
    stats["n_windowed"] = n_used
    promoter_sets[sheet] = prom
    count_tables[sheet] = counts
    load_stats.append(stats)

load_stats = pd.DataFrame(load_stats)
load_stats.columns = ["sheet", "rows in sheet", "no missing element", f"spacer={SPACER}", "in alignment window"]
print(load_stats.to_string(index=False))

## 4. 背景鹼基組成

In [ ]:
def genome_background(gb_path):
    record = next(SeqIO.parse(gb_path, "genbank"))
    seq = str(record.seq).upper()
    counts = np.array([seq.count(b) for b in BASES], dtype=float)
    return counts / counts.sum(), record


def flank_background(promoters, width):
    seqs = promoters["UP"].str[:width]
    joined = "".join(seqs)
    counts = np.array([joined.count(b) for b in BASES], dtype=float)
    return counts / counts.sum()


genome_bg, genome_record = genome_background(GENOME_GB)
print(f"{genome_record.id}  {genome_record.description}")
print(f"  length {len(genome_record.seq):,} bp   GC {genome_bg[1] + genome_bg[2]:.4f}")
print("  genome background: " + "  ".join(f"{b}={p:.5f}" for b, p in zip(BASES, genome_bg)))

if BACKGROUND_SOURCE == "genome":
    background = genome_bg
elif BACKGROUND_SOURCE == "flank":
    background = flank_background(promoter_sets[SHEETS[0]], FLANK_BG_WIDTH)
    print(f"\n  flank background (UP[:{FLANK_BG_WIDTH}] of {SHEETS[0]}): "
          + "  ".join(f"{b}={p:.5f}" for b, p in zip(BASES, background)))
else:
    raise ValueError(f"unknown BACKGROUND_SOURCE: {BACKGROUND_SOURCE!r}")

## 5. 每個位置的 information content 與 KL divergence

In [ ]:
def conservation_table(counts, background):
    freq = counts[BASES].to_numpy(dtype=float)
    freq = freq / freq.sum(axis=1, keepdims=True)

    log_f = np.zeros_like(freq)
    np.log2(freq, out=log_f, where=freq > 0)      # f = 0 的位置留 0，f·log f 也是 0
    entropy = -(freq * log_f).sum(axis=1)
    kl = (freq * (log_f - np.log2(background)[None, :])).sum(axis=1)

    out = pd.DataFrame({
        "rel_pos": counts["rel_pos"].to_numpy(),
        "n_acgt": counts["n_acgt"].to_numpy(),
        "entropy_bits": entropy,
        "IC_bits": 2.0 - entropy,
        "KL_bits": kl,
    })
    for i, b in enumerate(BASES):
        out[f"freq_{b}"] = freq[:, i]
    return out


per_position = {}
for sheet in SHEETS:
    cons = conservation_table(count_tables[sheet], background)
    merged = energy_mat.merge(cons, on="rel_pos", how="left", validate="one_to_one")
    merged.insert(0, "sheet", sheet)
    per_position[sheet] = merged

per_position_df = pd.concat(per_position.values(), ignore_index=True)

print("max |KL − IC| per sheet (E. coli 背景接近均勻，兩者幾乎相同):")
for sheet in SHEETS:
    d = per_position[sheet]
    print(f"  {sheet:8s} {np.abs(d['KL_bits'] - d['IC_bits']).max():.4f} bits")

print(f"\ntop conserved positions ({SHEETS[0]}):")
print(per_position[SHEETS[0]]
      .nlargest(10, "IC_bits")[["rel_pos", "element", "IC_bits", "KL_bits", "energy_absum"]]
      .round(3).to_string(index=False))


# --- 錨點自我驗證 ---
# −10 起點的定義是 m10_start = kernel #3 (−35 filter) 的 conv2 tap + M35_LEN + spacer。
# 只要座標錯位 1 bp，天然共識序列就讀不出 TATAAT / TTGACA，下面的 assert 會擋下來。
def consensus_seq(position_df, lo, hi):
    idx = position_df.set_index("rel_pos")
    return "".join(
        BASES[int(np.argmax([idx.at[r, f"freq_{b}"] for b in BASES]))]
        for r in range(lo, hi + 1)
    )


print("\nanchor self-check (每個 sheet 都必須讀出 TTGACA / TATAAT)")
for sheet in SHEETS:
    m35_seq = consensus_seq(per_position[sheet], -(SPACER + M35_LEN), -(SPACER + 1))
    m10_seq = consensus_seq(per_position[sheet], 0, M10_LEN - 1)
    print(f"  {sheet:8s} rel {-(SPACER + M35_LEN)}..{-(SPACER + 1)} = {m35_seq}   rel 0..{M10_LEN - 1} = {m10_seq}")
    assert m35_seq == "TTGACA", f"{sheet}: −35 錨點錯位，讀到 {m35_seq}"
    assert m10_seq == "TATAAT", f"{sheet}: −10 錨點錯位，讀到 {m10_seq}"

## 6. 每個 (位置, 鹼基) 的 logo letter height 與 log2 enrichment

- logo letter height：`f_b(i) · IC(i)`
- **log2 enrichment**（不加權）：`log2(f_b(i) / p_b)`——標準的 log-odds/enrichment
  score，純粹看「這個鹼基在這個位置比背景富集/貧化多少」，不受頻率大小影響。
  `f_b(i) = 0` 時無法定義（`log2(0)` 發散），該筆設為 `NaN`，下游相關係數/繪圖會自動略過。
- `kl_contrib`（頻率加權的 KL 貢獻，`f_b(i)·log2(f_b(i)/p_b)`，加總即為 `KL(i)`）仍保留在
  表裡供對照/驗證，但圖3改用不加權的 `log2_enrichment` 當 y 軸——因為加權後低頻率鹼基的
  訊號會被壓向 0，方向性反而不明顯。
- 模型端用**中心化** energy `e(i,b) − mean_b e(i,b)`，去掉每個位置的常數位移

In [ ]:
def build_per_base(position_df, background):
    records = []
    bg = dict(zip(BASES, background))
    for _, row in position_df.iterrows():
        energies = np.array([row[b] for b in BASES], dtype=float)
        centered = energies - energies.mean()
        for i, b in enumerate(BASES):
            f = float(row[f"freq_{b}"])
            records.append({
                "sheet": row["sheet"],
                "rel_pos": int(row["rel_pos"]),
                "element": row["element"],
                "base": b,
                "energy": float(energies[i]),
                "energy_centered": float(centered[i]),
                "freq": f,
                "letter_height": f * float(row["IC_bits"]),
                "kl_contrib": f * np.log2(f / bg[b]) if f > 0 else 0.0,
                "log2_enrichment": np.log2(f / bg[b]) if f > 0 else np.nan,
            })
    return pd.DataFrame(records)


per_base = {sheet: build_per_base(per_position[sheet], background) for sheet in SHEETS}
per_base_df = pd.concat(per_base.values(), ignore_index=True)

print(per_base_df.groupby("sheet").agg(
    n=("rel_pos", "size"),
    energy_centered_min=("energy_centered", "min"),
    energy_centered_max=("energy_centered", "max"),
    letter_height_max=("letter_height", "max"),
    kl_contrib_min=("kl_contrib", "min"),
    kl_contrib_max=("kl_contrib", "max"),
    log2_enrichment_nan=("log2_enrichment", lambda s: int(s.isna().sum())),
    log2_enrichment_min=("log2_enrichment", "min"),
    log2_enrichment_max=("log2_enrichment", "max"),
).round(3))

## 7. 相關係數彙總

### 用了哪些統計方法

每一組 (x, y) 都同時算兩種相關係數，理由跟算法各不相同：

- **Pearson r**：假設線性關係，對離群值敏感；p 值用 `scipy.stats.pearsonr` 內建的
  t 分布算（標準做法，n 不會太小時沒問題）。
- **Spearman ρ**：只看單調排序，不假設線性、對離群值/非線性較穩健；
  **p 值改用 permutation test**（`scipy.stats.permutation_test`，`permutation_type="pairings"`），
  不用 `scipy.stats.spearmanr` 內建的 p 值。原因：那個內建 p 值是用 t 分布近似算的，
  scipy 官方文件寫明「只在 n>500 時準確」；n 很小時（例如 −35／−10 在 position 層級
  只有 6 個位置）一旦 ρ=±1，近似公式的分母會變成 0，直接把 p 算成 `0.0`——那是近似公式
  在邊界失效的假象，不是真的「不可能發生」。改用 permutation test 後：n! 小於等於
  `PERM_RESAMPLES`（9999）時 scipy 會自動窮舉所有排列做 **exact test**（n=6 時
  6!=720 < 9999，符合這個條件，−35 這組現在算出 p=2.8e-03，不再是 0），n 較大時用
  9999 次隨機排列近似（見 `spearman_perm_pvalue()`）。
  > **解析度下限**：n 較大（例如圖3的 n=24/64/80）且相關很強時，9999 次隨機排列裡
  > 可能一次都沒有比觀測值更極端，這時印出來的 p 會卡在 `2/(9999+1) ≈ 2.0e-4`——
  > 這是「小於等於這個值」的上界，不是精確算出來的數字。真正的 p 可能更小，但沒必要
  > 為了這個再拉高 `PERM_RESAMPLES`，因為已經遠低於任何常見顯著門檻（0.05/0.01/0.001），
  > 不影響任何結論。
- **Bootstrap 95% CI**（只在逐元件的列算，`bootstrap_r_ci`，百分位法，5000 次重抽樣）：
  給 Pearson r 一個不確定性範圍，尤其 −35／−10 這種 n=6 的元件，r 本身很大但样本數很小，
  CI 可以看出實際上有多寬。
- **r_between／r_within 分解**（`decompose_by_base`，只套用在 per-(position, base) 層級）：
  不是教科書上有名字的檢定，是這個 notebook 自訂的拆解，用來回答
  「圖3看起來的分群，到底是純鹼基組成偏好，還是同一鹼基在不同位置上真的有被調節」：
  - `r_between`：只用 4 個鹼基的 (平均 x, 平均 y) 算 Pearson r → **純組成偏好**
  - `r_within`：每個點扣掉所屬鹼基的平均之後再算 Pearson r → **扣掉組成後剩下的位置專一資訊**

  兩者可以差很多。若 `r_within ≈ 0` 而 `r_between` 很高，代表模型在該元件只學到鹼基組成，
  沒有任何位置知識；圖3統計框上顯示的 pooled r/ρ/p 其實是這兩者混在一起的結果，
  不能直接當成「位置專一」的證據，要對照這裡的分解才知道。

In [ ]:
N_BOOT = 5000        # Pearson r 的 bootstrap 重抽樣次數
BOOT_SEED = 0
PERM_RESAMPLES = 9999  # Spearman p 的 permutation test 重抽樣次數（n! 夠小時 scipy 會自動做 exact test）
PERM_SEED = 0


def _pearson_rows(X, Y):
    """一次算完 (n_boot, n) 重抽樣矩陣每一列的 Pearson r；退化樣本回傳 nan。"""
    Xc = X - X.mean(axis=1, keepdims=True)
    Yc = Y - Y.mean(axis=1, keepdims=True)
    num = (Xc * Yc).sum(axis=1)
    den = np.sqrt((Xc ** 2).sum(axis=1) * (Yc ** 2).sum(axis=1))
    return np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan)


def bootstrap_r_ci(x, y, n_boot=N_BOOT, seed=BOOT_SEED, alpha=0.05):
    """Pearson r 的 bootstrap 百分位信賴區間。n 很小時退化的重抽樣會被濾掉。"""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    n = len(x)
    if n < 3:
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    rs = _pearson_rows(x[idx], y[idx])
    rs = rs[np.isfinite(rs)]
    if rs.size < 100:                     # 有效重抽樣太少，CI 沒有意義
        return np.nan, np.nan
    lo, hi = np.percentile(rs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)


def spearman_perm_pvalue(x, y, n_resamples=PERM_RESAMPLES, seed=PERM_SEED):
    """Spearman p 值改用 permutation test 算，不用 scipy spearmanr 內建的 t 分布近似。

    scipy 文件明講那個近似「只在 n>500 時準確」；n 很小時（例如 -35/-10 只有 6 個位置）
    在 rho=±1 會因為除以零直接算出 p=0.0，是近似公式失效的假象，不是真的顯著。
    這裡改用 scipy.stats.permutation_test：n! 小於等於 n_resamples 時會自動窮舉做
    exact test（例如 n=6 時 6!=720 < 9999，自動 exact），n 較大時用 n_resamples 次
    隨機排列近似（見 scipy permutation_test 文件的 n_resamples 說明）。
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 3:
        return np.nan

    def _stat(a):
        return spearmanr(a, y).statistic

    res = permutation_test((x,), _stat, permutation_type="pairings",
                            n_resamples=n_resamples, random_state=seed)
    return float(res.pvalue)


def correlate(x, y, n_boot=0):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    pr, pp = pearsonr(x, y)
    sr = spearmanr(x, y)
    sp = spearman_perm_pvalue(x, y)
    out = {"n": int(ok.sum()), "pearson_r": pr, "pearson_p": pp,
           "spearman_rho": float(sr.correlation), "spearman_p": sp}
    if n_boot:
        out["r_lo"], out["r_hi"] = bootstrap_r_ci(x, y, n_boot=n_boot)
    return out


def decompose_by_base(d, xcol="energy_centered", ycol="kl_contrib"):
    """把 (位置, 鹼基) 層級的相關拆成兩塊。

    r_between: 只用 4 個鹼基的平均值算相關 —— 純粹的組成偏好（例如「A/T 好、G/C 壞」）。
    r_within : 各自扣掉所屬鹼基的平均之後再算相關 —— 扣掉組成偏好後剩下的位置專一資訊。

    x/y 裡的 NaN（例如 log2_enrichment 在 freq=0 時無法定義）在算 r_within 前先過濾掉，
    否則 pearsonr 會整個算炸；r_between 用 groupby().mean() 本來就會自動跳過 NaN。
    """
    x = d[xcol].to_numpy(dtype=float)
    y = d[ycol].to_numpy(dtype=float)
    base_x = d.groupby("base")[xcol].mean().reindex(BASES).to_numpy()
    base_y = d.groupby("base")[ycol].mean().reindex(BASES).to_numpy()
    r_between = pearsonr(base_x, base_y)[0]

    resid_x = x - d.groupby("base")[xcol].transform("mean").to_numpy()
    resid_y = y - d.groupby("base")[ycol].transform("mean").to_numpy()
    ok = np.isfinite(resid_x) & np.isfinite(resid_y)
    r_within, p_within = pearsonr(resid_x[ok], resid_y[ok])

    ok_total = np.isfinite(x) & np.isfinite(y)
    return {"r_total": pearsonr(x[ok_total], y[ok_total])[0], "r_between": float(r_between),
            "r_within": float(r_within), "p_within": float(p_within)}


# 四個 (層級, x, y) 組合，後面的彙總與繪圖共用
METRIC_SPECS = [
    ("per position", "energy_absum", "IC_bits"),
    ("per position", "energy_absum", "KL_bits"),
    ("per (position, base)", "energy_centered", "letter_height"),
    ("per (position, base)", "energy_centered", "log2_enrichment"),
]


def _frame_for(sheet, level):
    return per_position[sheet] if level == "per position" else per_base[sheet]


rows = []
for sheet in SHEETS:
    # 既有的整體與 core 子集
    for subset_name, mask in [("all positions", None),
                              (f"core {CORE_RANGE[0]}..{CORE_RANGE[1]}", CORE_RANGE)]:
        for level, xcol, ycol in METRIC_SPECS:
            d = _frame_for(sheet, level)
            if mask is not None:
                d = d[d["rel_pos"].between(*mask)]
            rows.append({"sheet": sheet, "subset": subset_name, "level": level,
                         "x": xcol, "y": ycol, **correlate(d[xcol], d[ycol])})

    # 逐元件
    for element in ELEMENT_ORDER:
        for level, xcol, ycol in METRIC_SPECS:
            d = _frame_for(sheet, level)
            d = d[d["element"] == element]
            if len(d) < 3:
                continue
            row = {"sheet": sheet, "subset": f"element: {element}", "level": level,
                   "x": xcol, "y": ycol, **correlate(d[xcol], d[ycol], n_boot=N_BOOT)}
            if ycol == "log2_enrichment":
                row.update(decompose_by_base(d, xcol, ycol))
            rows.append(row)

correlation_df = pd.DataFrame(rows)
print(correlation_df.round(4).to_string(index=False))

print("\n" + "=" * 78)
print("base 層級的訊號拆解：組成偏好 (between) vs 位置專一 (within)")
print("  r_between = 只用 4 個鹼基的平均值，即『A/T 好、G/C 壞』這種純組成偏好")
print("  r_within  = 扣掉各鹼基平均後的殘差相關，即模型真正學到的位置專一資訊")
print("=" * 78)
for sheet in SHEETS:
    sub = correlation_df[(correlation_df["sheet"] == sheet)
                         & (correlation_df["y"] == "log2_enrichment")
                         & correlation_df["r_within"].notna()]
    sub = sub.assign(element=sub["subset"].str.replace("element: ", "", regex=False))
    print(f"\n{sheet}")
    print(sub[["element", "n", "r_total", "r_between", "r_within", "p_within"]]
          .to_string(index=False,
                     formatters={"r_total": "{:+.3f}".format,
                                 "r_between": "{:+.3f}".format,
                                 "r_within": "{:+.3f}".format,
                                 "p_within": "{:.2e}".format}))

## 8. 圖 1–3：依元件拆解

每張圖都是 **2 列（sheet）× 5 欄（元件）**：

- **圖 1／圖 2（Position vs Position）**：一個點 = 一個位置。x = 位置重要性，y = IC / KL。
  回答的是：模型有沒有把區辨力放在天然保守的位置上？
- **圖 3（Nucleotide vs Nucleotide）**：一個點 = (位置, 鹼基)。x = 中心化 energy，
  y = log2 enrichment（不加權的 `log2(f_b(i)/p_b)`，帶正負號）。
  回答的是**方向性**：模型偏好的鹼基跟天然富集/貧化的鹼基一不一致？

### 為什麼圖 1/2 取絕對值、圖 3 一定要留正負號

兩者的 x 軸其實是同一個量的兩個層級 —— **圖 1/2 的 x 就是圖 3 的 x 對四個鹼基取絕對值加總**
（`Σ_b |e(i,b) − mean_b| == energy_absum`，實測差距 3e-16）。差別在問的問題不同：

- **圖 1/2 問「多重要」**：位置的重要性沒有方向可言，兩軸（`energy_absum` 與 IC/KL）
  本來就都是非負的量綱，取絕對值是必要的。
- **圖 3 問「哪個方向」**：`x > 0` 代表模型偏好該鹼基、`x < 0` 代表排斥；
  `y > 0` 代表天然富集、`y < 0` 代表貧化。整張圖測的就是**兩邊方向一不一致**，
  取絕對值等於把要測的東西刪掉。

實測（Es.co，全部 256 點）：現行 `r = 0.614`；改成 `|x|` 掉到 **0.053**；
改成 `|y|` 變成 **−0.121**；兩個都取絕對值是 0.436，但那衡量的是「模型區辨強的地方
天然也偏離背景較多」—— 已經是圖 1/2 在講的事，只是換成逐鹼基granularity，沒有新資訊。

直觀的例子是 rel +1（−10 的 A，天然 94% 是 A）：模型 A 是 `+1.12`、C/G/T 是
`−0.32 ~ −0.43`；天然 A 是 `+1.94`、C/G/T 是 `−2.9 ~ −4.9`。一取 `|x|`，
A 和 C/G/T 全部擠到 x 大的那一端，「誰被偏好」的資訊就沒了。

### 位置重要性怎麼定義

用 **`energy_absum(i) = Σ_b |e(i,b) − mean_b e(i,·)|`**（置中後的絕對值總和），
不用 `max−min`。理由是 `max−min` 只看最極端的兩個鹼基，會把
`(A,T,G,C) = (+1,−1,0,0)` 和 `(+1,−1,+1,−1)` 判成一樣重要（兩者 span 都是 2），
但後者其實四個鹼基都被強烈推向某一邊，區辨力是前者的兩倍
（sum|centered| 分別為 2 與 4）。真實資料裡也有這種例子：rel −28（UP）的 span 只有
0.661，但 C（−0.468）與 G（−0.459）**兩個**都被強烈排斥，sum|centered| 達 1.211。

**一定要先置中。** 同一位置四個鹼基一起加常數 `c`，等於每條序列的 energy 都 +c，
會被 `conv2.bias` / `param_e0` 吸收，完全不影響模型對鹼基的區辨 —— 那是 gauge 自由度。
而 conv1 的原始權重並非置中（各位置平均值介於 −0.40 到 +0.04，64 個位置裡有 21 個
`|mean| > 0.10`），所以直接對原始權重取 `Σ|w|` 會把這個沒有資訊的位移算成「重要性」。

`energy_span`（max−min）與 `energy_sd`（L2 版本）仍保留在 per-position CSV 裡供對照。

版面規則：

- **每個子圖的 x/y 軸都完全獨立縮放**（`sharex=False, sharey=False`），
  為的是看清每個元件內部的相關結構。代價是**跨格比較座標的絕對大小沒有意義**，
  要比較強弱一律看統計框裡的數字
- x / y 軸標示各只在整張圖標一次（`fig.supxlabel` / `fig.supylabel`）；
  最左欄的 `Es.co` / `Es.co$` 是列標，指的是資料來源不是量綱

### 換成任意位置子集

欄位不一定要是那五個元件。`plot_by_element(..., groups=[...])` 可以指定任意子集，
版面與統計標註完全一樣。每筆 group 是 `(label, selector)` 或 `(label, selector, color)`，
selector 有四種寫法：

| 寫法 | 意義 |
|---|---|
| `"-10"` | 元件名（`groups=None` 時的預設行為就是這個） |
| `(-14, -7)` | `rel_pos` 區間，**含端點**，可跨元件 |
| `element_positions("UP", last=8)` | 明確位置清單 |
| `lambda d: d["freq_A"] > 0.5` | 自訂布林遮罩 |

`element_positions(element, first=N, last=N)` 取的是該元件在模型裡**實際有參數**的
`rel_pos`，不是 naive 的 range —— 這樣才不會踩到 −26/−25/−24 與 −15 那幾個未覆蓋的空隙。
例如 `element_positions("UP", last=8)` 回傳 −34…−27，剛好是 conv1 第 2 個 kernel 的
footprint（UP 共 16 個位置、中間無空隙，前 8 個 −42…−35 則是第 1 個 kernel）。

`rel_pos` 與 `element` 兩個欄位在 per-position 與 per-base 表裡都有，所以同一組 selector
兩個 `level` 都能用。函式回傳 `(fig, stats_df)`，`stats_df` 是每一格的 n / Pearson /
Spearman，可以直接存檔。可執行的範例寫在圖 3 後面那一格（預設註解掉）。
- 統計框標 `n`、**Pearson `r`／`p`** 與 **Spearman `ρ`／`p`** 兩組（`_stats_box`，都是對該
  子圖當下畫的那批點直接算的 pooled 統計量；兩種 p 值各自的算法見第 7 節「用了哪些統計
  方法」——Spearman 的 p 是 permutation test，不是 scipy 內建的 t 分布近似）。
  Pearson 假設線性關係、對離群值敏感；Spearman 只看單調排序、對離群值/非線性較穩健。
  兩者同時給，是為了能對照「線性擬合得好不好」跟「排序有沒有單調關係」是否一致。
  bootstrap 95% CI 都在 correlations CSV 裡（−35 與 −10 在 position 層級各只有 6 個位置，
  r ≈ 0.93–0.97 看起來很高，但 CI 分別是 [+0.91, +1.00] 與 [+0.79, +1.00]，要配合 CI 讀）
- **圖3（Nucleotide vs Nucleotide）的統計框要小心解讀**：那個 pooled r/ρ/p 是把「4 個
  鹼基之間的組成差異（分群）」跟「同一鹼基在不同位置上的細部趨勢（群內）」混在一起算的
  ——跟看圖時「這些點是不是清楚分成 4 群顏色」直覺對應的其實是 `r_between`
  （見下方 `decompose_by_base` 的 `r_between`/`r_within` 分解，只在 `correlation_df`
  裡，沒有畫在圖上）。DIS/downstream 就是典型例子：pooled `r_total = 0.274, p = 0.014`
  看起來顯著，但拆開後 `r_between = 0.758`、`r_within = −0.024, p = 0.83`——那個「顯著」
  幾乎完全是四個鹼基分群造成的，同一鹼基內部跟 energy 沒有關係。

### 四個實測結論（圖3改用 log2_enrichment 後重新確認，數字取 Es.co；Es.co$ 走向一致）

**梯度**：−35 ≈ −10 > spacer > UP ≫ DIS/downstream。
其中 −35 的 6 個位置在兩張 sheet 都達到 Spearman ρ = 1.000 —— 模型對 −35 各位置重要性的
**排序**與天然保守度完全一致（position 層級，不受這次 y 軸改動影響）。

**DIS/downstream 兩個層級同時判死刑。** position 層級 r = −0.16 / −0.10（不顯著），
base 層級 `r_total = 0.274`，拆開後 `r_within = −0.024, p = 0.83` ——
那點相關**幾乎完全**來自組成偏好（`r_between = 0.758`），位置專一成分是 0
（跟改用 log2_enrichment 之前的 `kl_contrib` 版本 `r_within = −0.036, p = 0.75` 結論一致）。
模型在該區有明顯的能量區辨（`energy_absum` 最高到 1.02），天然 E. coli 卻幾乎沒有保守性
（IC 最高 0.067），代表從合成 library 學到的 discriminator / ITS 偏好在天然 promoter
沒有對應的選擇壓力痕跡。

**UP 是 dissociation 的案例。** position 層級 r ≈ 0.27–0.36 且 CI 跨過 0 —— 但這不是模型
學不好，而是 UP 區 16 個位置的天然 IC 全部落在 0.002–0.045 bits，**根本沒有排序可以還原**。
到了 base 層級 `r_total = 0.603`，且扣掉組成偏好後仍有 `r_within = 0.428, p = 4.25e-4`。
也就是模型主要抓到 UP 的 AT-rich **組成**（`r_between = 0.879`），但不只如此
——跟改用 log2_enrichment 前幾乎同一組數字（`r_within = 0.437`）。

**spacer 意外地是位置專一的，而且比加權版本更明顯**：`r_within = 0.719 > r_between = 0.605`
（Es.co$：`0.755 > 0.572`），跟「spacer 只是個墊片」的直覺相反，也比舊版 `kl_contrib`
的 `0.708 > 0.628` 差距更大——顯示這個方向性訊號不是頻率加權造出來的假象。

**−35／−10 的 base 層級同時是最強的組成偏好與最強的位置專一**：`−35` 的
`r_between = 0.962, r_within = 0.912`；`−10` 的 `r_between = 0.984, r_within = 0.770`。
兩者的位置層級 r（0.93–0.97）本來就接近上限，base 層級把這個訊號進一步拆解後，
發現「哪個鹼基」跟「哪個位置」兩種資訊在這兩個元件都同時被模型學到。

> 註：`letter_height`（logo 字高 `f_b·IC`）恆為非負，對 energy 是非單調的，
> 不適合做散點，所以不畫圖；其相關係數仍保留在 correlations CSV 裡。

In [ ]:
def add_trendline(ax, x, y, color="black"):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3:
        return
    slope, intercept = np.polyfit(x[ok], y[ok], 1)
    xs = np.linspace(x[ok].min(), x[ok].max(), 50)
    ax.plot(xs, slope * xs + intercept, color=color, lw=1, ls="--", zorder=1)


def _format_p(p, threshold=0.001):
    """把 p 值印成純小數（不用科學記號）。小於 threshold 時印 '< 0.001'，
    因為極小的 p（例如 1e-10）用小數點展開會變成一長串 0，反而看不出資訊，
    低於這個門檻對「顯不顯著」的判讀也沒有差異；精確值仍留在 correlations CSV 裡。"""
    if not np.isfinite(p):
        return "p = n/a"
    if p < threshold:
        return f"p < {threshold:g}"
    return f"p = {p:.3f}"


def _stats_box(ax, x, y):
    st = correlate(x, y)
    ax.text(
        0.04, 0.96,
        f"n = {st['n']}\n"
        f"r = {st['pearson_r']:+.3f}, {_format_p(st['pearson_p'])}\n"
        f"ρ = {st['spearman_rho']:+.3f}, {_format_p(st['spearman_p'])}",
        transform=ax.transAxes, va="top", ha="left", fontsize=8, linespacing=1.35,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.88),
        zorder=5,
    )
    return st


NEUTRAL_GROUP_COLOR = "#555555"   # 跨越多個元件的自訂群組用的中性色


def element_positions(element, first=None, last=None):
    """某元件在模型裡『實際有參數』的 rel_pos，可取前 N 個或後 N 個。

    用實際位置而不是 range()，才不會被 −26/−25/−24 與 −15 那幾個未覆蓋的空隙坑到。
    例：element_positions("UP", last=8) -> [-34 … -27]，剛好是 conv1 第 2 個 kernel。
    """
    rel = sorted(energy_mat.loc[energy_mat["element"] == element, "rel_pos"].astype(int))
    if first is not None:
        rel = rel[:first]
    if last is not None:
        rel = rel[-last:]
    return rel


def _group_mask(df, selector):
    """把四種 selector 寫法統一成布林遮罩。"""
    if callable(selector):                                   # 自訂遮罩
        return np.asarray(selector(df), dtype=bool)
    if isinstance(selector, str):                            # 元件名
        return (df["element"] == selector).to_numpy()
    if (isinstance(selector, tuple) and len(selector) == 2   # (lo, hi) 區間，含端點
            and all(isinstance(v, (int, np.integer)) for v in selector)):
        return df["rel_pos"].between(selector[0], selector[1]).to_numpy()
    return df["rel_pos"].isin(list(selector)).to_numpy()     # 明確位置清單


def _group_color(sub, selector, explicit):
    if explicit is not None:
        return explicit
    if isinstance(selector, str) and selector in ELEMENT_COLORS:
        return ELEMENT_COLORS[selector]
    elements = set(sub["element"]) if len(sub) else set()
    if len(elements) == 1:
        return ELEMENT_COLORS.get(elements.pop(), NEUTRAL_GROUP_COLOR)
    return NEUTRAL_GROUP_COLOR


def plot_by_element(level, metric, ylabel, filename, groups=None,
                    panel_w=2.75, panel_h=3.15, min_fig_w=7.0):
    """2 列 (sheet) x N 欄的拆解圖。level: 'position' | 'perbase'

    groups: 欄位定義，None 時就是原本的五個元件。每筆為 (label, selector) 或
    (label, selector, color)，selector 支援四種寫法：

        "-10"                       元件名
        (-14, -7)                   rel_pos 區間，含端點
        element_positions("UP", last=8)   明確位置清單
        lambda d: d.freq_A > 0.5    自訂布林遮罩

    回傳 (fig, stats_df)，stats_df 是每一格的 n / Pearson / Spearman。
    """
    xcol = "energy_absum" if level == "position" else "energy_centered"
    # position 層級問「多重要」→ 絕對值加總；perbase 層級問「哪個方向」→ 必須保留正負號
    xlabel = (r"$\sum |ε(i,b)-\overline{ε(i)}|$"
              if level == "position" else
              r"$ε(i,b)-\overline{ε(i)}$")
    analysis_name = "Position vs Position" if level == "position" else "Nucleotide vs Nucleotide"

    if groups is None:
        groups = [(e, e) for e in ELEMENT_ORDER]
    groups = [(g[0], g[1], g[2] if len(g) > 2 else None) for g in groups]

    # 每個子圖的 x/y 軸都完全獨立縮放，方便看清每個元件內部的結構。
    # 只有 1~2 欄時 panel_w 放不下 18pt 的 suptitle，所以總寬度給 min_fig_w 保底。
    fig, axes = plt.subplots(
        len(SHEETS), len(groups),
        figsize=(max(panel_w * len(groups), min_fig_w), panel_h * len(SHEETS)),
        dpi=150, sharex=False, sharey=False, squeeze=False,
    )

    stats_rows = []
    for row, sheet in enumerate(SHEETS):
        source = per_position[sheet] if level == "position" else per_base[sheet]
        for col, (label, selector, color) in enumerate(groups):
            ax = axes[row][col]
            d = source[_group_mask(source, selector)]
            group_color = _group_color(d, selector, color)

            if d.empty:
                ax.set_axis_off()
                if row == 0:
                    ax.set_title(f"{label}\n0 points", fontsize=12, color=group_color)
                continue

            if level == "position":
                ax.scatter(d[xcol], d[metric], s=34, alpha=0.9,
                           color=group_color, edgecolors="white", linewidths=0.5,
                           zorder=3)
            else:
                for b in BASES:
                    sub = d[d["base"] == b]
                    ax.scatter(sub[xcol], sub[metric], s=20, alpha=0.75,
                               color=BASE_COLORS[b], edgecolors="none", label=b, zorder=3)
                ax.axvline(0, color="0.88", lw=0.8, zorder=0)

            # 少於 3 點時 pearsonr 會拋錯，只標點數
            if len(d.dropna(subset=[xcol, metric])) >= 3:
                add_trendline(ax, d[xcol], d[metric])
                st = _stats_box(ax, d[xcol], d[metric])
            else:
                ax.text(0.04, 0.96, f"n = {len(d)}", transform=ax.transAxes,
                        va="top", ha="left", fontsize=8,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="0.8", alpha=0.88),
                        zorder=5)
                st = {"n": len(d)}
            stats_rows.append({"sheet": sheet, "group": label, "level": level,
                               "x": xcol, "y": metric, **st})

            ax.axhline(0, color="0.88", lw=0.8, zorder=0)
            ax.tick_params(labelsize=10)

            # if level == "position":
            #     # 點少的時候把所有位置標出來，點多就只標 y 最大的兩個
            #     marked = d if len(d) <= 8 else d.nlargest(2, metric)
            #     for _, r in marked.iterrows():
            #         rel = int(r["rel_pos"])
            #         ax.annotate("0" if rel == 0 else f"{rel:+d}", (r[xcol], r[metric]),
            #                     textcoords="offset points", xytext=(5, 3), fontsize=6,
            #                     color="0.25", zorder=4)

            if row == 0:
                ax.set_title(f"{label}\n{len(d)} points", fontsize=12, color=group_color)
            # 最左欄只標 sheet 名稱；量綱由 fig.supylabel 統一標一次
            if col == 0:
                ax.set_ylabel(sheet, fontsize=10)

    # 上方留白給統計框，避免蓋到資料點。軸不共用，所以每一格都要各自調。
    for row in range(len(SHEETS)):
        for col in range(len(groups)):
            if not axes[row][col].get_visible() or not axes[row][col].has_data():
                continue
            lo, hi = axes[row][col].get_ylim()
            axes[row][col].set_ylim(lo, lo + (hi - lo) * 1.38)

    if level == "perbase":
        axes[0][-1].legend(frameon=False, fontsize=8, title="base", title_fontsize=8,
                           loc="upper left", bbox_to_anchor=(1.03, 1.0), borderaxespad=0)

    # x / y 軸標示各只出現一次
    fig.supxlabel(xlabel, fontsize=12)
    fig.supylabel(ylabel, fontsize=12)
    fig.suptitle(
        f"{analysis_name} analysis of {ylabel} and Energy",
        fontsize=18,
    )
    fig.tight_layout()
    for ext in ("svg", "png"):
        fig.savefig(FIG_DIR / f"{filename}.{ext}", bbox_inches="tight")
    plt.show()
    return fig, pd.DataFrame(stats_rows)


# 圖 1
plot_by_element("position", "IC_bits", "IC (bits)",
                f"EnergyVsIC_position_byElement_Ecoli_sp{SPACER}");

In [ ]:
# 圖 2
plot_by_element("position", "KL_bits", "KL divergence (bits)",
                f"EnergyVsKL_position_byElement_Ecoli_sp{SPACER}");

In [ ]:
# 圖 3
plot_by_element("perbase", "log2_enrichment", "Log2 enrichment (bits)",
                f"EnergyVsLog2Enrichment_perbase_byElement_Ecoli_sp{SPACER}");

In [ ]:
# === 自訂子集圖（預設全部註解掉，要用時取消註解即可）===
#
# plot_by_element 的 groups 參數可以把欄位換成任意位置子集，版面與統計標註完全一樣。
# 回傳的第二個值是每一格的 n / Pearson / Spearman，可以直接存檔。
#
# 例 1：把 UP 拆成兩個 conv1 kernel 各一欄（前 8 個 = −42…−35，後 8 個 = −34…−27）
# fig, stats = plot_by_element(
#     "position", "IC_bits", "IC (bits)",
#     f"EnergyVsIC_position_UPsplit_Ecoli_sp{SPACER}",
#     groups=[("UP 16",  element_positions("UP", last=16)),
#             ("UP last 9",  element_positions("UP", last=9)),
#             ("UP last 10", element_positions("UP", last=10)),
#             ],
# )
# display(stats)
#
# 例 2：只畫 UP 倒數 8 個鹼基（8 個點）。單欄時總寬度會自動吃 min_fig_w 保底
# plot_by_element("position", "IC_bits", "IC (bits)",
#                 f"EnergyVsIC_position_UPlast8_Ecoli_sp{SPACER}",
#                 groups=[("UP last 8 (-34..-27)", element_positions("UP", last=8))]);
#
# 例 3：任意 rel_pos 區間（含端點），可以跨元件
# plot_by_element("position", "IC_bits", "IC (bits)",
#                 f"EnergyVsIC_position_custom_Ecoli_sp{SPACER}",
#                 groups=[("-35 + 前半 spacer", (-23, -7)),
#                         ("-10 + discriminator", (0, 9))]);
#
# 例 4：同樣的 groups 也能用在 perbase 層級
# plot_by_element("perbase", "log2_enrichment", "Log2 enrichment (bits)",
#                 f"EnergyVsLog2Enrichment_perbase_UPlast8_Ecoli_sp{SPACER}",
#                 groups=[("UP last 8", element_positions("UP", last=8))]);
#
# 例 5：自訂遮罩（callable），例如只看天然 A/T 佔比高的位置
# plot_by_element("position", "IC_bits", "IC (bits)",
#                 f"EnergyVsIC_position_ATrich_Ecoli_sp{SPACER}",
#                 groups=[("AT-rich 位置", lambda d: d["freq_A"] + d["freq_T"] > 0.6)]);

## 9. 匯出

In [ ]:
tag = f"Ecoli_sp{SPACER}_{BACKGROUND_SOURCE}bg"
paths = {
    "per position": OUT_DIR / f"energy_vs_conservation_{tag}_positions.csv",
    "per (position, base)": OUT_DIR / f"energy_vs_conservation_{tag}_perbase.csv",
    "correlations": OUT_DIR / f"energy_vs_conservation_{tag}_correlations.csv",
}
per_position_df.to_csv(paths["per position"], index=False)
per_base_df.to_csv(paths["per (position, base)"], index=False)
correlation_df.to_csv(paths["correlations"], index=False)

for label, p in paths.items():
    print(f"{label:22s} -> {p}")
print()
for name in sorted(FIG_DIR.glob(f"EnergyVs*_Ecoli_sp{SPACER}*.*")):
    print(f"figure                 -> {name}")